In [0]:
%python
!pip install Faker
from data import generate_messy_data

In [0]:
%python
df = generate_messy_data()
spark_df = spark.createDataFrame(df)
spark_df.write.format('delta').option('mergeSchema', 'true').mode('overwrite').saveAsTable('transform.sql_db.sales_raw')

- convert pandas dataframe to spark
- write.format('delta') : writing new datafram in delta format
- mode("overwrite") : delete and replace incoming dataframe/table with whatever is existent
- saveAsTable("name of table")


**Bronze Transformation** 
- Add row id
- trimming spaces
 
- converting column names to lowercase

- renaming problematic columns

In [0]:
select * from _sqldf

In [0]:
select * from transform.sql_db.sales_silver;



In [0]:
CREATE TABLE IF NOT EXISTS transform.sql_db.sales_bronze
LIKE transform.sql_db.sales_raw;

-- Add row_id column
ALTER TABLE transform.sql_db.sales_bronze
ADD COLUMN row_id INT;

-- Insert data with transformations: trim spaces, lowercase column names, rename problematic columns, add row_id
INSERT INTO transform.sql_db.sales_bronze
SELECT
  CAST(user_id AS DOUBLE) AS user_id,
  trim(lower(first_name)) AS first_name,
  trim(lower(last_name)) AS last_name,
  trim(lower(email_addr)) AS email_addr,
  trim(lower(region_name)) AS region_name,
  CAST(sale_date AS DATE) AS sale_date,
  base_price,
  jan_sales,
  feb_sales,
  user_rating,
  batch_id,
  ingestion_timestamp,
  monotonically_increasing_id() AS row_id
FROM transform.sql_db.sales_raw;


In [0]:
select * from transform.sql_db.sales_bronze

In [0]:
-- we don't need to do any transformation in the bronze table because they pass the data as it is


**create silver table**


1. Enforce schema(This ensures every column has the correct type.)
2. Standardize formats (lowercase/uppercase,trim whitespace,standardize date formats,consistent column naming)
3. Remove duplicates
4. Fix null values
5. Validate data (emails, ranges, etc.)
6. Normalize data
7. Join datasets

In [0]:
--delete from transform.sql_db.sales_bronze;
drop table if exists transform.sql_db.sales_silver

In [0]:
-- create Silver Table and Enforce Schemas
CREATE OR REPLACE TABLE transform.sql_db.sales_silver AS
SELECT
  CAST(user_id AS LONG) AS user_id,
  CAST(first_name AS STRING) AS first_name,
  CAST(last_name AS STRING) AS last_name,
  CAST(email_addr AS STRING) AS email_addr,
  CAST(region_name AS STRING) AS region_name,
  CAST(sale_date AS DATE) AS sale_date,
  CAST(base_price AS DECIMAL(10,2)) AS base_price,
  CAST(jan_sales AS LONG) AS jan_sales,
  CAST(feb_sales AS LONG) AS feb_sales,
  CAST(user_rating AS LONG) AS user_rating,
  batch_id,
  ingestion_timestamp,
  row_id,
  current_timestamp() AS silver_processed_timestamp
FROM transform.sql_db.sales_bronze;

In [0]:

UPDATE transform.sql_db.sales_silver 
SET
  base_price = least(base_price, 1000),
  user_rating = least(greatest(user_rating,0), 5);



In [0]:
with dup_record as (
  select *, row_number() over(
    partition by user_id, first_name, last_name, email_addr, region_name, sale_date, base_price, jan_sales, feb_sales, user_rating
    ORDER BY row_id
  ) as dup_count
  from transform.sql_db.sales_silver
)
select * from dup_record where dup_count > 1

In [0]:
DELETE FROM transform.sql_db.sales_silver
WHERE row_id IN (
  SELECT row_id FROM (
    SELECT row_id, row_number() OVER (
      PARTITION BY user_id, first_name, last_name, email_addr, region_name, sale_date, base_price, jan_sales, feb_sales, user_rating
      ORDER BY row_id 
    ) AS dup_cnt
    FROM transform.sql_db.sales_silver
  ) t
  WHERE dup_cnt > 1
)

In [0]:
SELECT
  regexp_replace(
    regexp_replace(
      regexp_replace(
        regexp_replace(email_addr, '#', ''),
        '%', ''),
      '#example', '@example'),
    '_com', '.com'
  ) AS email_addr
FROM transform.sql_db.sales_silver

In [0]:
UPDATE transform.sql_db.sales_silver
SET email_addr= 
  regexp_replace(
    regexp_replace(
      regexp_replace(
        regexp_replace(email_addr, '#', ''),
        '%', ''),
      '#example', '@example'),
    '_com', '.com'
  ) 

In [0]:
select email_addr FROM transform.sql_db.sales_silver
WHERE email_addr NOT LIKE '%@example.com';

In [0]:
UPDATE transform.sql_db.sales_silver
SET email_addr = regexp_replace(email_addr, 'example.com$', '@example.com')
WHERE email_addr NOT LIKE '%@%';

In [0]:
UPDATE transform.sql_db.sales_silver
SET email_addr =
  concat(replace(email_addr, 'example.com', ''), '@example.com')
  WHERE email_addr LIKE '%example.com' 
  AND email_addr NOT LIKE '%@%';

In [0]:
select email_addr, concat(replace(email_addr, '@.com', ''), '@example.com') as new_email FROM transform.sql_db.sales_silver
WHERE email_addr LIKE '%@.com'; 
  --AND email_addr NOT LIKE '%@%';

select email_addr, concat(replace(email_addr, '@example', ''), '@example.com') as new_email FROM transform.sql_db.sales_silver
WHERE email_addr LIKE '%@example' 


In [0]:
UPDATE transform.sql_db.sales_silver
SET email_addr =
  concat(replace(email_addr, '@.com', ''), '@example.com')
  WHERE email_addr LIKE '%@.com' 

In [0]:
UPDATE transform.sql_db.sales_silver
SET email_addr =
  concat(replace(email_addr, '@example', ''), '@example.com')
  WHERE email_addr LIKE '%@example' 

In [0]:
SELECT 
  email_addr AS original_email,
  concat(replace(email_addr, 'example.com', ''), '@example.com') AS cleaned_email
FROM transform.sql_db.sales_silver
WHERE email_addr LIKE '%example.com' 
  AND email_addr NOT LIKE '%@%';

In [0]:
SELECT email_addr FROM transform.sql_db.sales_silver
WHERE email_addr LIKE '%example.com' AND email_addr NOT LIKE '%@example.com';

In [0]:
select email_addr from transform.sql_db.sales_silver 
WHERE email_addr NOT LIKE '%@%'

In [0]:
SELECT email_addr AS original_email,
  concat(replace(email_addr, '.com', ''), '@example.com') AS cleaned_email FROM transform.sql_db.sales_silver
WHERE email_addr NOT LIKE '%.com%'


In [0]:
UPDATE transform.sql_db.sales_silver
SET email_addr =
  concat(replace(email_addr, '.com', ''), '@example.com') 
WHERE email_addr NOT LIKE '%.com%'

In [0]:
UPDATE transform.sql_db.sales_silver
SET email_addr =
  concat(replace(email_addr, '@.com', ''), '@example.com') 
WHERE email_addr NOT LIKE '%@example%'

# **FIXING NULLS**

In [0]:
SELECT first_name,last_name,email_addr, concat(first_name, last_name, '@example.com') AS remove_null_email
FROM transform.sql_db.sales_silver
WHERE email_addr IS NULL

In [0]:
SELECT s.first_name, s.last_name, s.email_addr, ss.first_name, ss.last_name, ss.email_addr
FROM transform.sql_db.sales_silver s
LEFT JOIN transform.sql_db.sales_silver ss ON s.first_name = ss.first_name AND s.last_name = ss.last_name
WHERE s.first_name IS NULL OR s.last_name IS NULL OR s.email_addr IS NULL

In [0]:
with fixed_columns as  (
  SELECT 
  email_addr,
  COALESCE(first_name, SPLIT_PART(email_addr, last_name, 1)) AS first_name_fixed,   COALESCE(last_name, 
    nullif(SPLIT_PART(SPLIT_PART(email_addr, '@', 1), first_name, 2), '')
  ) AS last_name_fixed, first_name, last_name
FROM transform.sql_db.sales_silver
-- This part ensures we only focus on rows that actually need help
WHERE first_name IS NULL or last_name IS NULL or email_addr IS NULL
)
select first_name_fixed, last_name_fixed, concat(first_name_fixed, last_name_fixed, '@example.com') AS email_fixed, email_addr from fixed_columns







In [0]:
UPDATE transform.sql_db.sales_silver
SET 
  first_name = COALESCE(first_name, SPLIT_PART(email_addr, last_name, 1)),
  last_name = COALESCE(last_name, nullif(SPLIT_PART(SPLIT_PART(email_addr, '@', 1), first_name, 2), '')),
  email_addr = CONCAT(
      COALESCE(first_name, SPLIT_PART(email_addr, last_name, 1)),
      COALESCE(last_name, nullif(SPLIT_PART(SPLIT_PART(email_addr, '@', 1), first_name, 2), '')),
      '@example.com'
  )
WHERE first_name IS NULL OR last_name IS NULL OR email_addr IS NULL

In [0]:
SELECT 
  -- We generate a new ID from scratch to fill all gaps
  ROW_NUMBER() OVER (ORDER BY row_id) AS user_id_fixed,
  user_id,
  email_addr
FROM transform.sql_db.sales_silver
--ORDER BY row_id

# **Validation**

In [0]:
SELECT * FROM transform.sql_db.sales_silver WHERE first_name IS NULL OR last_name IS NULL OR email_addr IS NULL

In [0]:
create view transform.sql_db.sales_silver_outlier AS
SELECT * FROM transform.sql_db.sales_silver 
WHERE 
  first_name IS NULL 
OR 
  last_name IS NULL 
OR
  email_addr IS NULL 
OR 
  user_rating < 0 
OR 
  user_rating > 5 
OR 
  base_price > 1000
OR
  region_name NOT IN ('north', 'south', 'east', 'west', 'unknown','central')

In [0]:
SELECT * FROM transform.sql_db.sales_silver WHERE email_addr NOT LIKE '%@%.%'

In [0]:
delete FROM transform.sql_db.sales_silver
WHERE 
  first_name IS NULL 
OR 
  last_name IS NULL 
OR
  email_addr IS NULL 
OR 
  user_rating < 0 
OR 
  user_rating > 5 
OR 
  base_price > 1000

In [0]:
select region_name, concat(first_name,' ',last_name) as full_name FROM transform.sql_db.sales_silver
where region_name IN ('north', 'south', 'east', 'west', 'unknown','central')

In [0]:
SELECT * FROM transform.sql_db.sales_silver WHERE region_name IS NULL 

In [0]:
SELECT email_addr, COUNT(*) AS cnt FROM transform.sql_db.sales_silver
GROUP BY email_addr
HAVING cnt > 1;

SELECT * FROM transform.sql_db.sales_silver WHERE email_addr NOT LIKE '%@%.%';

In [0]:
CREATE OR REPLACE VIEW transform.sql_db.sales_silver_north AS
SELECT * FROM transform.sql_db.sales_silver WHERE region_name = 'north';

CREATE OR REPLACE VIEW transform.sql_db.sales_silver_south AS
SELECT * FROM transform.sql_db.sales_silver WHERE region_name = 'south';

CREATE OR REPLACE VIEW transform.sql_db.sales_silver_east AS
SELECT * FROM transform.sql_db.sales_silver WHERE region_name = 'east';

CREATE OR REPLACE VIEW transform.sql_db.sales_silver_west AS
SELECT * FROM transform.sql_db.sales_silver WHERE region_name = 'west';

CREATE OR REPLACE VIEW transform.sql_db.sales_silver_unknown AS
SELECT * FROM transform.sql_db.sales_silver WHERE region_name = 'unknown';

CREATE OR REPLACE VIEW transform.sql_db.sales_silver_central AS
SELECT * FROM transform.sql_db.sales_silver WHERE region_name = 'central';

In [0]:
ALTER TABLE transform.sql_db.sales_silver
ADD CONSTRAINT not_null_name CHECK (first_name IS NOT NULL AND last_name IS NOT NULL);

ALTER TABLE transform.sql_db.sales_silver
ADD CONSTRAINT valid_user_rating CHECK (user_rating >= 0 AND user_rating <= 5);

ALTER TABLE transform.sql_db.sales_silver
ADD CONSTRAINT strict_example_email CHECK (email_addr LIKE '%@example.com');

ALTER TABLE transform.sql_db.sales_silver
ALTER COLUMN region_name SET NOT NULL;

-- this is assuming we have a max price of 1000
ALTER TABLE transform.sql_db.sales_silver
ADD CONSTRAINT price_cap CHECK (base_price <= 1000);

-- Remove valid_sale_date constraint as it is currently blocking due to violations

ALTER TABLE transform.sql_db.sales_silver
ADD CONSTRAINT region_enum CHECK (region_name IN ('north', 'south', 'east', 'west', 'unknown', 'central'));


In [0]:
SHOW TBLPROPERTIES transform.sql_db.sales_silver;

In [0]:
SELECT * FROM transform.sql_db.sales_silver
WHERE region_name not IN ('west', 'east', 'south', 'north', 'unknown', 'central')

In [0]:
ALTER TABLE transform.sql_db.sales_silver SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name');
alter table transform.sql_db.sales_silver drop column row_id;

In [0]:
select * from transform.sql_db.sales_bronze